In [8]:
import mysql.connector as connector
from functools import wraps
import pandas as pd

In [ ]:
def with_db(f):
    @wraps(f)
    def wrapper(*args, **kwargs):
        # Setup connection
        password = ""

        conn = connector.connect(host="localhost", port=3306, user="root", password=password) 
        try:
            # Pass the connection as the first argument to the function
            return f(conn, *args, **kwargs)
        finally:
            conn.close()
    return wrapper

In [7]:
@with_db
def show_db(conn):
    query1 = "use LittleLemon;"
    query2 = "show tables;"
    with conn.cursor() as cursor:
        cursor.execute(query1)
        cursor.execute(query2)

        for row in cursor.fetchall():
            print(row)

show_db()

('Addresses',)
('Bookings',)
('Customer',)
('Menu',)
('MenuCourses',)
('MenuCuisines',)
('MenuDesserts',)
('MenuDrinks',)
('MenuStarters',)
('OrderDeliveryStatus',)
('OrderStatus',)
('Orders',)
('OrdersView',)
('Staff',)


In [16]:
@with_db
def get_join_tables(conn):
    query1 = "use LittleLemon;"
    query2 = """
    select c.CustName, c.CustPhone, c.CustEmail, a.Pincode, a.City, a.State, a.Country, o.TotalCost
    from (select * from Orders where TotalCost >= 60) as o
    join Customer c on o.CustomerId = c.CustomerId
    join Addresses a on c.AddressId = a.AddressId
    order by o.TotalCost;
    """
    results = []
    with conn.cursor() as cursor:
        cursor.execute(query1)
        cursor.execute(query2)

        results = cursor.fetchall()
        column_names = [desc[0] for desc in cursor.description]
    
    return pd.DataFrame.from_records(results, columns=column_names)

get_join_tables()

,CustName,CustPhone,CustEmail,Pincode,City,State,Country,TotalCost
0,Customer_23,54569203,Cust23@customer23.com,38424,City30,StateName,CountryName,62
1,Customer_34,71664347,Cust34@customer34.com,67205,City34,StateName,CountryName,66
2,Customer_41,36382067,Cust41@customer41.com,5427,City36,StateName,CountryName,66
3,Customer_19,80194595,Cust19@customer19.com,55454,City47,StateName,CountryName,76
4,Customer_30,47702358,Cust30@customer30.com,39140,City42,StateName,CountryName,78
5,Customer_8,56470361,Cust8@customer8.com,37937,City26,StateName,CountryName,81
6,Customer_47,1530540,Cust47@customer47.com,61702,City43,StateName,CountryName,83
7,Customer_31,74617266,Cust31@customer31.com,56593,City4,StateName,CountryName,87
8,Customer_46,68453640,Cust46@customer46.com,66703,City32,StateName,CountryName,101
9,Customer_30,47702358,Cust30@customer30.com,39140,City42,StateName,CountryName,105
